In [2]:
import numpy as np
import os

# Paths
data_root = "E:/project/Data"
output_root = "E:/project/WeedCube_Patches"
os.makedirs(output_root, exist_ok=True)

# Crops you currently have
crops = ["canola", "soybean", "waterhemp"]

# Patch size
PATCH_SIZE = 64
TRAIN_RATIO = 0.7  # 70% train, 30% test

def extract_patches(cube, patch_size=64, stride=64):
    """Extract patches with stride (can be < patch_size for overlap)."""
    H, W, B = cube.shape
    patches = []
    
    # Pad cube so dimensions are multiples of patch_size
    pad_h = (patch_size - H % patch_size) % patch_size
    pad_w = (patch_size - W % patch_size) % patch_size
    cube = np.pad(cube, ((0,pad_h),(0,pad_w),(0,0)), mode='reflect')

    # Extract patches
    for i in range(0, cube.shape[0] - patch_size + 1, stride):
        for j in range(0, cube.shape[1] - patch_size + 1, stride):
            patch = cube[i:i+patch_size, j:j+patch_size, :]
            patches.append(patch)
    return patches


for crop in crops:
    crop_path = os.path.join(data_root, crop)
    files = [f for f in os.listdir(crop_path) if f.endswith(".npy")]

    # Split cubes into train/test (by file, not pixels)
    split_idx = int(len(files) * TRAIN_RATIO)
    train_files = files[:split_idx]
    test_files = files[split_idx:]

    for split, file_list in [("train", train_files), ("test", test_files)]:
        split_path = os.path.join(output_root, split, crop)
        os.makedirs(split_path, exist_ok=True)

        for file in file_list:
            cube = np.load(os.path.join(crop_path, file))
            patches = extract_patches(cube)

            # Save patches
            for idx, patch in enumerate(patches):
                save_name = f"{file.replace('.npy','')}_patch{idx}.npy"
                np.save(os.path.join(split_path, save_name), patch)

            print(f"{crop} {split}: {file} → {len(patches)} patches")


canola train: canola_1.npy → 120 patches
canola train: canola_10.npy → 120 patches
canola train: canola_11.npy → 112 patches
canola train: canola_12.npy → 120 patches
canola train: canola_13.npy → 128 patches
canola train: canola_14.npy → 128 patches
canola train: canola_15.npy → 128 patches
canola train: canola_16.npy → 120 patches
canola train: canola_17.npy → 120 patches
canola train: canola_18.npy → 128 patches
canola train: canola_19.npy → 128 patches
canola train: canola_2.npy → 112 patches
canola train: canola_20.npy → 120 patches
canola train: canola_3.npy → 112 patches
canola test: canola_4.npy → 112 patches
canola test: canola_5.npy → 120 patches
canola test: canola_6.npy → 105 patches
canola test: canola_7.npy → 105 patches
canola test: canola_8.npy → 98 patches
canola test: canola_9.npy → 120 patches


In [5]:
import numpy as np
import os

data_root = "E:/project/Data"
crops = ["canola", "soybean", "waterhemp"]

for crop in crops:
    crop_path = os.path.join(data_root, crop)
    npy_files = []
    # Walk through all subfolders
    for root, dirs, files in os.walk(crop_path):
        for file in files:
            if file.endswith(".npy"):
                npy_files.append(os.path.join(root, file))
    
    if not npy_files:
        print(f"--- Crop: {crop} --- No .npy files found.")
    else:
        print(f"\n--- Crop: {crop} ---")
        for f in npy_files:
            cube = np.load(f)
            print(f"{os.path.basename(f)} → shape: {cube.shape}")




--- Crop: canola ---
canola_1.npy → shape: (493, 926, 224)
canola_10.npy → shape: (499, 914, 224)
canola_11.npy → shape: (424, 966, 224)
canola_12.npy → shape: (498, 926, 224)
canola_13.npy → shape: (506, 968, 224)
canola_14.npy → shape: (503, 963, 224)
canola_15.npy → shape: (512, 964, 224)
canola_16.npy → shape: (511, 916, 224)
canola_17.npy → shape: (512, 946, 224)
canola_18.npy → shape: (486, 964, 224)
canola_19.npy → shape: (451, 968, 224)
canola_2.npy → shape: (449, 872, 224)
canola_20.npy → shape: (492, 956, 224)
canola_3.npy → shape: (472, 850, 224)
canola_4.npy → shape: (497, 869, 224)
canola_5.npy → shape: (457, 951, 224)
canola_6.npy → shape: (414, 919, 224)
canola_7.npy → shape: (445, 907, 224)
canola_8.npy → shape: (426, 896, 224)
canola_9.npy → shape: (455, 931, 224)

--- Crop: soybean ---
soybean_1.npy → shape: (433, 955, 224)
soybean_10.npy → shape: (435, 1000, 224)
soybean_11.npy → shape: (435, 1011, 224)
soybean_12.npy → shape: (430, 1002, 224)
soybean_13.npy → shape

In [6]:
import numpy as np
import os

# Paths
data_root = "E:/project/Data"
output_root = "E:/project/WeedCube_Patches"
os.makedirs(output_root, exist_ok=True)

crops = ["canola", "soybean", "waterhemp"]

PATCH_SIZE = 64
TRAIN_RATIO = 0.7

def pad_cube(cube, patch_size=PATCH_SIZE):
    """Pad cube so dimensions are multiples of patch_size"""
    H, W, B = cube.shape
    pad_h = (patch_size - H % patch_size) % patch_size
    pad_w = (patch_size - W % patch_size) % patch_size
    cube = np.pad(cube, ((0,pad_h),(0,pad_w),(0,0)), mode='reflect')
    return cube

def extract_patches(cube, patch_size=PATCH_SIZE, stride=PATCH_SIZE):
    """Extract patches with stride (default = non-overlapping)"""
    H, W, B = cube.shape
    patches = []
    for i in range(0, H - patch_size + 1, stride):
        for j in range(0, W - patch_size + 1, stride):
            patch = cube[i:i+patch_size, j:j+patch_size, :]
            patches.append(patch)
    return patches

for crop in crops:
    crop_path = os.path.join(data_root, crop)
    files = [f for f in os.listdir(crop_path) if f.endswith(".npy")]

    split_idx = int(len(files) * TRAIN_RATIO)
    train_files = files[:split_idx]
    test_files = files[split_idx:]

    for split, file_list in [("train", train_files), ("test", test_files)]:
        split_path = os.path.join(output_root, split, crop)
        os.makedirs(split_path, exist_ok=True)

        for file in file_list:
            cube = np.load(os.path.join(crop_path, file))
            cube = pad_cube(cube)  # pad before patching
            patches = extract_patches(cube)

            for idx, patch in enumerate(patches):
                save_name = f"{file.replace('.npy','')}_patch{idx}.npy"
                np.save(os.path.join(split_path, save_name), patch)

            print(f"{crop} {split}: {file} → {len(patches)} patches")


canola train: canola_1.npy → 120 patches
canola train: canola_10.npy → 120 patches
canola train: canola_11.npy → 112 patches
canola train: canola_12.npy → 120 patches
canola train: canola_13.npy → 128 patches
canola train: canola_14.npy → 128 patches
canola train: canola_15.npy → 128 patches
canola train: canola_16.npy → 120 patches
canola train: canola_17.npy → 120 patches
canola train: canola_18.npy → 128 patches
canola train: canola_19.npy → 128 patches
canola train: canola_2.npy → 112 patches
canola train: canola_20.npy → 120 patches
canola train: canola_3.npy → 112 patches
canola test: canola_4.npy → 112 patches
canola test: canola_5.npy → 120 patches
canola test: canola_6.npy → 105 patches
canola test: canola_7.npy → 105 patches
canola test: canola_8.npy → 98 patches
canola test: canola_9.npy → 120 patches


In [7]:
import numpy as np
import os

# Paths
data_root = "E:/project/Data"
output_root = "E:/project/WeedCube_Patches"
os.makedirs(output_root, exist_ok=True)

# Only fix soybean & waterhemp (leave canola untouched)
crop_paths = {
    "soybean": os.path.join(data_root, "soybean", "soybean"),
    "waterhemp": os.path.join(data_root, "waterhemp", "waterhemp")
}

PATCH_SIZE = 64
TRAIN_RATIO = 0.7

def pad_cube(cube, patch_size=PATCH_SIZE):
    H, W, B = cube.shape
    pad_h = (patch_size - H % patch_size) % patch_size
    pad_w = (patch_size - W % patch_size) % patch_size
    cube = np.pad(cube, ((0,pad_h),(0,pad_w),(0,0)), mode='reflect')
    return cube

def extract_patches(cube, patch_size=PATCH_SIZE, stride=PATCH_SIZE):
    H, W, B = cube.shape
    patches = []
    for i in range(0, H - patch_size + 1, stride):
        for j in range(0, W - patch_size + 1, stride):
            patch = cube[i:i+patch_size, j:j+patch_size, :]
            patches.append(patch)
    return patches

for crop, crop_path in crop_paths.items():
    files = [f for f in os.listdir(crop_path) if f.endswith(".npy")]

    split_idx = int(len(files) * TRAIN_RATIO)
    train_files = files[:split_idx]
    test_files = files[split_idx:]

    for split, file_list in [("train", train_files), ("test", test_files)]:
        split_path = os.path.join(output_root, split, crop)
        os.makedirs(split_path, exist_ok=True)

        for file in file_list:
            cube = np.load(os.path.join(crop_path, file))
            cube = pad_cube(cube)
            patches = extract_patches(cube)

            for idx, patch in enumerate(patches):
                save_name = f"{file.replace('.npy','')}_patch{idx}.npy"
                np.save(os.path.join(split_path, save_name), patch)

            print(f"{crop} {split}: {file} → {len(patches)} patches")


soybean train: soybean_1.npy → 105 patches
soybean train: soybean_10.npy → 112 patches
soybean train: soybean_11.npy → 112 patches
soybean train: soybean_12.npy → 112 patches
soybean train: soybean_13.npy → 112 patches
soybean train: soybean_14.npy → 112 patches
soybean train: soybean_15.npy → 112 patches
soybean train: soybean_16.npy → 112 patches
soybean train: soybean_17.npy → 105 patches
soybean train: soybean_18.npy → 112 patches
soybean train: soybean_19.npy → 112 patches
soybean train: soybean_2.npy → 112 patches
soybean train: soybean_20.npy → 112 patches
soybean train: soybean_3.npy → 112 patches
soybean test: soybean_4.npy → 112 patches
soybean test: soybean_5.npy → 112 patches
soybean test: soybean_6.npy → 112 patches
soybean test: soybean_7.npy → 112 patches
soybean test: soybean_8.npy → 112 patches
soybean test: soybean_9.npy → 112 patches
waterhemp train: waterhemp_1.npy → 128 patches
waterhemp train: waterhemp_10.npy → 128 patches
waterhemp train: waterhemp_11.npy → 112 

In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
